In [1]:
import os
import shutil
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

In [2]:
## REMOVE PATIENTS FROM DATASET
# Current list is list of t2w


def is_bad_file(filename, bad_list):
    for code in bad_list:
        if code in filename:
            return True
    return False

def copy_and_remove_bad_files(root_dir, bad_list, copy_dir):
    os.makedirs(copy_dir, exist_ok=True)

    for filename in os.listdir(root_dir):
        
        if filename.endswith(".png") and "LP-" in filename:
            if not is_bad_file(filename, bad_list):
                print(filename)
                src_path = os.path.join(root_dir, filename)
                dst_path = os.path.join(copy_dir, filename)
                shutil.copy2(src_path, dst_path)


    return copy_dir

# List of patients with t2w
bad_list = ["LP-0009","LP-0010","LP-0085","LP-0117","LP-0216","LP-0576","LP-0080","LP-0136","LP-0239","LP-0337","LP-0343","LP-0367","LP-0568","LP-0998"]

# root dir with current dataset
root_dir = "E:\MRgLITT Data\\new data\\2021-12-21\segmentationData_3Regions"
# directory to copy updated patients into
root_copy = "E:\MRgLITT Data\\new data\\2021-12-21\\segmentationData_3Regions_no_t2"
copy_and_remove_bad_files(root_dir, bad_list, root_copy)

In [157]:
## CONVERT SEGMENTATION PNG TO 3 CHANNEL

def convert_to_3channel(seg_dir):

    # Convert seg png to np array
    seg = Image.open(seg_dir)
    seg_np = np.array(seg)

    # Stack np array to create 51x51x3
    seg3d = np.stack([seg_np] * 3, axis=-1)
    
    # Return as PIL Image
    return(Image.fromarray(seg3d))

#####################################

root_dir_1d = "E:\MRgLITT Data\\new data\\2021-12-21\segmentationData_3Regions_no_t2"
final_dir__3d = "E:\MRgLITT Data\\new data\\2021-12-21\segmentationData_3Regions_no_t2_3D"

for filename in os.listdir(root_dir_1d):
    if filename.lower().endswith(".png"):
        file_path = os.path.join(root_dir_1d, filename)
        # print("Current png:", file_path)
        
        # Convert 51x51x1 to 51x51x3
        seg_png = convert_to_3channel(file_path)
        output_path = os.path.join(final_dir__3d, filename)

        # Save image
        seg_png.save(output_path)

In [158]:
##COMBINE ANATOMICAL MRI AND SEGMENTED MRI
#problem: to convert to gray scale causes the 51x51x3 at the end to flatten to 51x51x1
# the images are still overlayed, but not 3D

def convert_to_mixedChannel(seg_dir, anatomical_dir):

    # Convert seg png to np array
    seg = Image.open(seg_dir)
    seg_np = np.array(seg)

    # Convert seg png to np array
    anatomical = Image.open(anatomical_dir)
    anat_np = np.array(anatomical)

    # Replace the second channel of anat_np with seg_np
    mixed_np = anat_np
    mixed_np[:, :, 1] = seg_np

    return(Image.fromarray(mixed_np).convert("L"))



# folder with segmentation pngs
seg_1d = "E:\MRgLITT Data\\new data\\2021-12-21\segmentationData_3Regions_no_t2"
# folder with anatomical pngs
anatomical_dir = "E:\MRgLITT Data\\new data\\2021-12-21\\anatomicalProbesEye_Corrected"
# folder to copy mixed pngs into
new_dir ="E:\MRgLITT Data\\new data\\2021-12-21\segmentationData_3Regions_no_t2_1D"


seg_files = os.listdir(seg_1d)
anatomical_files = os.listdir(anatomical_dir)
combined_filenames = zip(seg_files, anatomical_files)

# Iterate through both anatomical and segmented MRIs
for filename_seg, filename_anat in combined_filenames:
    file_path_seg = os.path.join(seg_1d, filename_seg)
    file_path_anat = os.path.join(anatomical_dir, filename_anat)
        
    # Convert 51x51x1 to 51x51x3
    mixed_png = convert_to_mixedChannel(file_path_seg, file_path_anat)
    output_path = os.path.join(new_dir, filename_anat)

    # Save image
    mixed_png.save(output_path)


In [35]:
# Delete duplicate files (if any)

def find_duplicate_filenames(folder_path):
    # Create a dictionary to store filenames and their occurrences
    filename_dict = {}
    duplicate_files = []

    # Iterate over the files in the folder
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".png"):
            # If the file is a PNG file, check if its name is already in the dictionary
            if filename in filename_dict:
                # If the filename is already present, add it to the duplicate_files list
                duplicate_files.append(filename)
            else:
                # If the filename is not present, add it to the dictionary with a count of 1
                filename_dict[filename] = 1

    return duplicate_files

if __name__ == "__main__":
    file_path = "E:\MRgLITT Data\\new data\\2021-12-21\segmentationData_allData"
    duplicate_files = find_duplicate_filenames(file_path)

    if duplicate_files:
        print("Duplicate files found:")
        for filename in duplicate_files:
            print(filename)
    else:
        print("No duplicate files found.")



No duplicate files found.
